In [11]:
"""
A Yariverse corporate credit rating scorecard.

This is an educational & illustrative model inspired by the general structure of
published rating agency methodologies (weighted quantitative factor grid ->
composite score -> letter rating).

Rating scale uses a broad category with no numeric modifiers:
    Aaa > Aa > A > Baa > Ba > B > Caa > Ca > C
    (Aaa-Baa = investment grade, Ba-C = speculative grade)

Auto path detection: run from anywhere; outputs land next to this script.
Methogology Overview 
1. Five financial ratios are computed per company: Debt/EBITDA (leverage),
   EBITDA/Interest (coverage), EBITDA Margin (profitability), FFO/Debt
   (cash flow adequacy), and Revenue Growth.
2. Each ratio is scored 1 (Aaa, strongest) to 9 (C, weakest) via a breakpoint
   table (see score_leverage / score_coverage / score_margin /
   score_ffo_to_debt / score_growth below).
3. The five scores are combined into a Quantitative Weighted Score using
   SECTOR-SPECIFIC weights (different industries emphasize different ratios).
4. The quantitative score is blended with a Business Profile Score (a
   qualitative, analyst-supplied 1-9 input see section 2b below) and then
   a notching adjustment is applied for structural/legal/eventrisk factors.

What the model doesn't capsture real world limitations
This scorecard is a simplified, educational approximation. It differs from
how any major rating agency actually rates issuers in several important ways:
  - Qualitative judgment is only a single manual input (business_profile_score),
    not genuine analyst assessment of management quality, competitive
    position, governance, or country/industry risk.
  - Sector grids (Industrial / Utility / Tech-Growth) are illustrative
    presets, not real published sector specific rating factor grids,
    which vary far more granularly (by sub industry) and are proprietary (legally and company specific).
  - Breakpoint thresholds are reasonable, commonly cited ranges for corporate
    credit analysis and not unpublished threshold tables.
  - Notching here is a single manual -2..+2 adjustment; real notching covers
    subordination, guarantees, parent/subsidiary support, covenant strength,
    and structural seniority in far more granular, rules based ways.
  - No forward looking adjustment: real ratings often reflect where an
    analyst expects trends to go, not just trailing financials.
  - No default/transition rate calibration: real letter ratings are
    calibrated historically to observed default rates per category; this
    model's letters are just a rank ordering label, not probability calibrated.

Bottom line: useful for learning the mechanics of a rating scorecard, ot a substitute for an actual credit rating and
shouldn't inform real investment or lending decisions.
"""

"\nA Yariverse corporate credit rating scorecard.\n\nThis is an educational & illustrative model inspired by the general structure of\npublished rating agency methodologies (weighted quantitative factor grid ->\ncomposite score -> letter rating).\n\nRating scale uses a broad category with no numeric modifiers:\n    Aaa > Aa > A > Baa > Ba > B > Caa > Ca > C\n    (Aaa-Baa = investment grade, Ba-C = speculative grade)\n\nAuto path detection: run from anywhere; outputs land next to this script.\nMethogology Overview \n1. Five financial ratios are computed per company: Debt/EBITDA (leverage),\n   EBITDA/Interest (coverage), EBITDA Margin (profitability), FFO/Debt\n   (cash flow adequacy), and Revenue Growth.\n2. Each ratio is scored 1 (Aaa, strongest) to 9 (C, weakest) via a breakpoint\n   table (see score_leverage / score_coverage / score_margin /\n   score_ffo_to_debt / score_growth below).\n3. The five scores are combined into a Quantitative Weighted Score using\n   SECTOR-SPECIFIC weig

In [3]:
#importing libraries
from __future__ import annotations
import os
import sys
from dataclasses import dataclass, field
from typing import List, Dict
import pandas as pd

In [4]:
# Auto path detection
try:
    SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    SCRIPT_DIR = os.getcwd()
OUTPUT_CSV = os.path.join(SCRIPT_DIR, "credit_rating_results.csv")
OUTPUT_HTML = os.path.join(SCRIPT_DIR, "credit_rating_dashboard.html")

In [5]:
#Rating scale
RATING_SCALE: Dict[int, str] = {
    1: "Aaa", 2: "Aa", 3: "A", 4: "Baa",
    5: "Ba", 6: "B", 7: "Caa", 8: "Ca", 9: "C",
}
INVESTMENT_GRADE_CUTOFF = 4  # scores 1-4 (Aaa..Baa) are investment grade

In [6]:
RATING_COLOR: Dict[str, str] = {
    "Aaa": "#f7c6d9", "Aa": "#f4b8cf", "A": "#f2a9c5",
    "Baa": "#ef9bbb", "Ba": "#e37fa8", "B": "#d96a97",
    "Caa": "#c85486", "Ca": "#b8406f", "C": "#9c2f5b",
}

In [7]:
#Preset factor weights (must sum to 1.00)
#    Real rating agencies use different weighting grids per industry (a utility
#    is not scored like a bank or a tech company). Below are 3 illustrative
#    sector presets so the model at least nods at that reality.
SECTOR_WEIGHTS: Dict[str, Dict[str, float]] = {
    "Industrial": {
        "leverage": 0.30, "coverage": 0.25, "margin": 0.20,
        "ffo_to_debt": 0.15, "growth": 0.10,
    },
    "Utility": {  # utilities: leverage tolerated more, stability/coverage weighted up
        "leverage": 0.20, "coverage": 0.30, "margin": 0.15,
        "ffo_to_debt": 0.25, "growth": 0.10,
    },
    "Tech / Growth": {  # growth and margin matter more, leverage typically lower anyway
        "leverage": 0.20, "coverage": 0.20, "margin": 0.25,
        "ffo_to_debt": 0.15, "growth": 0.20,
    },
}
DEFAULT_SECTOR = "Industrial"
FACTOR_WEIGHTS: Dict[str, float] = SECTOR_WEIGHTS[DEFAULT_SECTOR]
for _sector, _weights in SECTOR_WEIGHTS.items():
    assert abs(sum(_weights.values()) - 1.0) < 1e-9, f"{_sector} weights must sum to 1.00"

In [8]:
#Qualitative overlay, what the quant only model can't see
#
#    Real ratings blend a quantitative grid with analyst qualitative judgment
#    (competitive position, management/governance, industry risk) and then
#    apply "notching" for structural/legal factors (subordination, covenants,
#    parent/guarantor support) and event risk (M&A, litigation, refinancing
#    cliffs). This model represents that as: blended_score = QUANT_WEIGHT * quantitative_score

In [9]:
#      final_score   = clip(1, 9, round(blended_score) + notching_adjustment)
#
#    business_profile_score (1-9, analyst assigned) and notching_adjustment
#    (-2..+2, analyst-assigned) are inputs the model has no way to derive them from financials alone, exactly like in real rating work.
QUANT_WEIGHT = 0.70
QUALITATIVE_WEIGHT = 0.30
assert abs(QUANT_WEIGHT + QUALITATIVE_WEIGHT - 1.0) < 1e-9

# Preset scoring formulas (breakpoint tables), formulas used identically in the Excel workbook via nested IFS()

def score_leverage(debt_to_ebitda: float) -> int:
    """Lower Debt/EBITDA = stronger credit = lower (better) score."""
    if debt_to_ebitda <= 1.0:
        return 1
    elif debt_to_ebitda <= 2.0:
        return 2
    elif debt_to_ebitda <= 3.0:
        return 3
    elif debt_to_ebitda <= 4.0:
        return 4
    elif debt_to_ebitda <= 5.0:
        return 5
    elif debt_to_ebitda <= 6.5:
        return 6
    elif debt_to_ebitda <= 8.0:
        return 7
    elif debt_to_ebitda <= 10.0:
        return 8
    else:
        return 9


def score_coverage(ebitda_to_interest: float) -> int:
    """Higher EBITDA/Interest = stronger credit = lower (better) score."""
    if ebitda_to_interest >= 15:
        return 1
    elif ebitda_to_interest >= 10:
        return 2
    elif ebitda_to_interest >= 7:
        return 3
    elif ebitda_to_interest >= 5:
        return 4
    elif ebitda_to_interest >= 3.5:
        return 5
    elif ebitda_to_interest >= 2:
        return 6
    elif ebitda_to_interest >= 1:
        return 7
    elif ebitda_to_interest >= 0.5:
        return 8
    else:
        return 9


def score_margin(ebitda_margin: float) -> int:
    """EBITDA margin as a fraction (0.20 = 20%). Higher = better."""
    if ebitda_margin >= 0.35:
        return 1
    elif ebitda_margin >= 0.28:
        return 2
    elif ebitda_margin >= 0.22:
        return 3
    elif ebitda_margin >= 0.17:
        return 4
    elif ebitda_margin >= 0.12:
        return 5
    elif ebitda_margin >= 0.08:
        return 6
    elif ebitda_margin >= 0.04:
        return 7
    elif ebitda_margin >= 0.0:
        return 8
    else:
        return 9


def score_ffo_to_debt(ffo_to_debt: float) -> int:
    """FFO/Debt as a fraction. Higher = better."""
    if ffo_to_debt >= 0.60:
        return 1
    elif ffo_to_debt >= 0.45:
        return 2
    elif ffo_to_debt >= 0.35:
        return 3
    elif ffo_to_debt >= 0.25:
        return 4
    elif ffo_to_debt >= 0.17:
        return 5
    elif ffo_to_debt >= 0.10:
        return 6
    elif ffo_to_debt >= 0.05:
        return 7
    elif ffo_to_debt >= 0.0:
        return 8
    else:
        return 9


def score_growth(revenue_growth: float) -> int:
    """YoY revenue growth as a fraction. Higher = better."""
    if revenue_growth >= 0.15:
        return 1
    elif revenue_growth >= 0.10:
        return 2
    elif revenue_growth >= 0.07:
        return 3
    elif revenue_growth >= 0.04:
        return 4
    elif revenue_growth >= 0.02:
        return 5
    elif revenue_growth >= 0.0:
        return 6
    elif revenue_growth >= -0.03:
        return 7
    elif revenue_growth >= -0.08:
        return 8
    else:
        return 9


#Company data model
@dataclass
class Company:
    name: str
    revenue: float
    prior_revenue: float
    ebitda: float
    interest_expense: float
    total_debt: float
    taxes: float
    sector: str = DEFAULT_SECTOR                # which weighting grid to use
    business_profile_score: int = 5             # 1 (best) - 9 (worst); analyst judgment input
    notching_adjustment: int = 0                # -2..+2; structural/legal/event risk overlay
    notching_reason: str = ""                   # e.g. "subordinated debt", "strong parent support"

    #derived ratios
    @property
    def debt_to_ebitda(self) -> float:
        return self.total_debt / self.ebitda if self.ebitda else float("inf")

    @property
    def ebitda_to_interest(self) -> float:
        return self.ebitda / self.interest_expense if self.interest_expense else float("inf")

    @property
    def ebitda_margin(self) -> float:
        return self.ebitda / self.revenue if self.revenue else 0.0

    @property
    def ffo_to_debt(self) -> float:
        ffo = self.ebitda - self.interest_expense - self.taxes
        return ffo / self.total_debt if self.total_debt else 0.0

    @property
    def revenue_growth(self) -> float:
        return (self.revenue - self.prior_revenue) / self.prior_revenue if self.prior_revenue else 0.0


#Composite scoring

def composite_score(c: Company) -> Dict[str, float]:
    weights = SECTOR_WEIGHTS.get(c.sector, FACTOR_WEIGHTS)

    s_lev = score_leverage(c.debt_to_ebitda)
    s_cov = score_coverage(c.ebitda_to_interest)
    s_mar = score_margin(c.ebitda_margin)
    s_ffo = score_ffo_to_debt(c.ffo_to_debt)
    s_gro = score_growth(c.revenue_growth)

    quant_weighted = (
        s_lev * weights["leverage"]
        + s_cov * weights["coverage"]
        + s_mar * weights["margin"]
        + s_ffo * weights["ffo_to_debt"]
        + s_gro * weights["growth"]
    )

    # Blend quantitative grid with the qualitative business profile input, then apply structural/event risk notching. This is the step real agencies do with analyst judgment that a purely financial ratio model can't replicate.
    blended = QUANT_WEIGHT * quant_weighted + QUALITATIVE_WEIGHT * c.business_profile_score
    final_score = min(9, max(1, round(blended) + c.notching_adjustment))
    rating = RATING_SCALE[final_score]
    grade = "Investment Grade" if final_score <= INVESTMENT_GRADE_CUTOFF else "Speculative Grade"

    return {
        "Company": c.name,
        "Sector": c.sector,
        "Debt/EBITDA (x)": round(c.debt_to_ebitda, 2),
        "EBITDA/Interest (x)": round(c.ebitda_to_interest, 2),
        "EBITDA Margin (%)": round(c.ebitda_margin * 100, 1),
        "FFO/Debt (%)": round(c.ffo_to_debt * 100, 1),
        "Revenue Growth (%)": round(c.revenue_growth * 100, 1),
        "Leverage Score": s_lev,
        "Coverage Score": s_cov,
        "Margin Score": s_mar,
        "FFO/Debt Score": s_ffo,
        "Growth Score": s_gro,
        "Quant Weighted Score": round(quant_weighted, 2),
        "Business Profile Score": c.business_profile_score,
        "Blended Score": round(blended, 2),
        "Notching Adj.": c.notching_adjustment,
        "Notching Reason": c.notching_reason,
        "Final Rating": rating,
        "Rating Grade": grade,
    }


#Sample portfolio (replace with real / loan book data)
SAMPLE_COMPANIES: List[Company] = [
    Company("Alpha Industrials", revenue=4200, prior_revenue=3900, ebitda=1050,
            interest_expense=90, total_debt=1800, taxes=150, sector="Industrial",
            business_profile_score=3, notching_adjustment=0, notching_reason=""),
    Company("Beacon Retail Co.", revenue=2600, prior_revenue=2550, ebitda=260,
            interest_expense=70, total_debt=1400, taxes=25, sector="Industrial",
            business_profile_score=6, notching_adjustment=0, notching_reason=""),
    Company("Cascade Utilities", revenue=5100, prior_revenue=4950, ebitda=2050,
            interest_expense=310, total_debt=7200, taxes=180, sector="Utility",
            business_profile_score=3, notching_adjustment=1, notching_reason="regulated monopoly, strong parent support"),
    Company("Delta Tech Corp", revenue=3300, prior_revenue=2870, ebitda=990,
            interest_expense=40, total_debt=600, taxes=140, sector="Tech / Growth",
            business_profile_score=3, notching_adjustment=0, notching_reason=""),
    Company("Everline Energy", revenue=1800, prior_revenue=1950, ebitda=270,
            interest_expense=95, total_debt=2100, taxes=5, sector="Utility",
            business_profile_score=6, notching_adjustment=-1, notching_reason="subordinated debt at opco"),
    Company("Foxglove Media", revenue=900, prior_revenue=980, ebitda=60,
            interest_expense=55, total_debt=850, taxes=0, sector="Tech / Growth",
            business_profile_score=7, notching_adjustment=0, notching_reason=""),
    Company("Granite Materials", revenue=2200, prior_revenue=2100, ebitda=440,
            interest_expense=60, total_debt=1300, taxes=45, sector="Industrial",
            business_profile_score=4, notching_adjustment=0, notching_reason=""),
    Company("Harbor Airlines", revenue=6100, prior_revenue=5600, ebitda=550,
            interest_expense=210, total_debt=4800, taxes=10, sector="Industrial",
            business_profile_score=6, notching_adjustment=1, notching_reason="event risk: pending litigation settlement"),
]


def build_results(companies: List[Company] = SAMPLE_COMPANIES) -> pd.DataFrame:
    rows = [composite_score(c) for c in companies]
    df = pd.DataFrame(rows).sort_values("Blended Score").reset_index(drop=True)
    df.insert(0, "Rank", df.index + 1)
    return df


# Baby pink HTML dashboard export
def render_html_dashboard(df: pd.DataFrame, path: str = OUTPUT_HTML) -> None:
    def rating_badge(rating: str) -> str:
        color = RATING_COLOR.get(rating, "#f2a9c5")
        return f'<span class="badge" style="background:{color}">{rating}</span>'

    rows_html = []
    for _, r in df.iterrows():
        notch = r['Notching Adj.']
        notch_display = f"{'+' if notch > 0 else ''}{notch}" + (f" ({r['Notching Reason']})" if r['Notching Reason'] else "")
        rows_html.append(f"""
        <tr>
          <td>{int(r['Rank'])}</td>
          <td class="company">{r['Company']}</td>
          <td>{r['Sector']}</td>
          <td>{r['Debt/EBITDA (x)']}</td>
          <td>{r['EBITDA/Interest (x)']}</td>
          <td>{r['EBITDA Margin (%)']}%</td>
          <td>{r['FFO/Debt (%)']}%</td>
          <td>{r['Revenue Growth (%)']}%</td>
          <td>{r['Quant Weighted Score']}</td>
          <td>{r['Business Profile Score']}</td>
          <td>{notch_display}</td>
          <td>{rating_badge(r['Final Rating'])}</td>
          <td>{r['Rating Grade']}</td>
        </tr>""")

    ig_count = int((df["Rating Grade"] == "Investment Grade").sum())
    sg_count = int((df["Rating Grade"] == "Speculative Grade").sum())
    avg_score = round(df["Blended Score"].mean(), 2)

    html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>Yariverse's Credit Rating Dashboard</title>
<style>
  :root {{
    --pink-bg: #fff0f5;
    --pink-panel: #ffe4ef;
    --pink-accent: #f2a9c5;
    --pink-deep: #d96a97;
    --pink-text: #7a3b52;
  }}
  * {{ box-sizing: border-box; }}
  body {{
    margin: 0;
    font-family: 'Segoe UI', Arial, sans-serif;
    background: var(--pink-bg);
    color: var(--pink-text);
    padding: 32px;
  }}
  h1 {{
    color: var(--pink-deep);
    margin-bottom: 4px;
  }}
  .subtitle {{
    color: var(--pink-text);
    opacity: 0.75;
    margin-top: 0;
    margin-bottom: 24px;
  }}
  .summary-cards {{
    display: flex;
    gap: 16px;
    margin-bottom: 28px;
    flex-wrap: wrap;
  }}
  .card {{
    background: var(--pink-panel);
    border: 1px solid var(--pink-accent);
    border-radius: 14px;
    padding: 18px 24px;
    min-width: 160px;
    box-shadow: 0 2px 6px rgba(217,106,151,0.15);
  }}
  .card .label {{
    font-size: 12px;
    text-transform: uppercase;
    letter-spacing: 0.06em;
    opacity: 0.7;
  }}
  .card .value {{
    font-size: 26px;
    font-weight: 700;
    color: var(--pink-deep);
  }}
  table {{
    width: 100%;
    border-collapse: collapse;
    background: white;
    border-radius: 14px;
    overflow: hidden;
    box-shadow: 0 2px 10px rgba(217,106,151,0.12);
  }}
  thead th {{
    background: var(--pink-accent);
    color: white;
    text-align: left;
    padding: 12px 14px;
    font-size: 13px;
    text-transform: uppercase;
    letter-spacing: 0.04em;
  }}
  tbody td {{
    padding: 11px 14px;
    border-bottom: 1px solid #fbdce9;
    font-size: 14px;
  }}
  tbody tr:nth-child(even) {{ background: #fff7fa; }}
  tbody tr:hover {{ background: #ffe4ef; }}
  td.company {{ font-weight: 600; color: var(--pink-deep); }}
  .badge {{
    display: inline-block;
    padding: 4px 12px;
    border-radius: 999px;
    color: white;
    font-weight: 700;
    font-size: 13px;
    min-width: 42px;
    text-align: center;
  }}
  .legend {{
    margin-top: 20px;
    font-size: 12px;
    color: var(--pink-text);
    opacity: 0.75;
  }}
  footer {{
    margin-top: 30px;
    font-size: 12px;
    opacity: 0.6;
  }}
  .methodology {{
    margin-top: 32px;
    background: var(--pink-panel);
    border: 1px solid var(--pink-accent);
    border-radius: 14px;
    padding: 20px 26px;
  }}
  .methodology h2 {{
    color: var(--pink-deep);
    font-size: 17px;
    margin-top: 0;
  }}
  .methodology p, .methodology li {{
    font-size: 13px;
    line-height: 1.5;
  }}
  .methodology ul {{
    margin: 8px 0;
    padding-left: 20px;
  }}
</style>
</head>
<body>
  <h1>Yariverse's Credit Rating Dashboard</h1>
  <p class="subtitle">Weighted quantitative scorecard &middot; Educational / practice model (not official rating factors)</p>

  <div class="summary-cards">
    <div class="card"><div class="label">Companies Rated</div><div class="value">{len(df)}</div></div>
    <div class="card"><div class="label">Investment Grade</div><div class="value">{ig_count}</div></div>
    <div class="card"><div class="label">Speculative Grade</div><div class="value">{sg_count}</div></div>
    <div class="card"><div class="label">Avg. Blended Score</div><div class="value">{avg_score}</div></div>
  </div>

  <table>
    <thead>
      <tr>
        <th>#</th><th>Company</th><th>Sector</th><th>Debt/EBITDA</th><th>EBITDA/Interest</th>
        <th>EBITDA Margin</th><th>FFO/Debt</th><th>Rev. Growth</th>
        <th>Quant Score</th><th>Bus. Profile</th><th>Notching</th><th>Rating</th><th>Grade</th>
      </tr>
    </thead>
    <tbody>
      {''.join(rows_html)}
    </tbody>
  </table>

  <p class="legend">Scale: 1&ndash;9 (Aaa best &rarr; C worst) &middot; Investment grade = score &le; 4 (Aaa/Aa/A/Baa) &middot; Speculative grade = score &ge; 5 (Ba/B/Caa/Ca/C) &middot; Final = 70% Quant Score + 30% Business Profile, then notching adjustment applied</p>

  <div class="methodology">
    <h2>Methodology &amp; Limitations</h2>
    <p><strong>How it's calculated:</strong> five ratios (leverage, coverage, margin, FFO/Debt, growth) are each
    scored 1&ndash;9 via breakpoint tables, combined with sector-specific weights into a Quantitative Score, then
    blended 70/30 with a manually-supplied Business Profile score (qualitative judgment) and adjusted by a
    notching factor for structural/legal/event-risk considerations before mapping to a letter rating.</p>
    <p><strong>What this model does not capture, versus a real Moody's rating:</strong></p>
    <ul>
      <li>Qualitative judgment is a single manual input here, not full analyst assessment of management,
      governance, competitive position, or country risk.</li>
      <li>Sector weighting grids are illustrative presets, not real, far more granular, proprietary
      sector methodologies.</li>
      <li>Breakpoint thresholds are reasonable educational approximations, not Moody's actual published
      (or unpublished) factor tables.</li>
      <li>Notching is a simple &plusmn;2 manual adjustment, not the full rules-based treatment of subordination,
      guarantees, and parent/subsidiary support used in practice.</li>
      <li>No forward-looking adjustment and no default-rate calibration behind the letter grades.</li>
    </ul>
    <p><strong>Use case:</strong> learning the mechanics of a rating scorecard / coursework &amp; portfolio practice.
    Not a substitute for an actual credit rating; not for real investment or lending decisions.</p>
  </div>

  <footer>Generated by credit_rating_engine.py</footer>
</body>
</html>"""
    with open(path, "w", encoding="utf-8") as f:
        f.write(html)


#Main
def main():
    df = build_results()
    df.to_csv(OUTPUT_CSV, index=False)
    render_html_dashboard(df)
    print(df.to_string(index=False))
    print(f"\nSaved: {OUTPUT_CSV}")
    print(f"Saved: {OUTPUT_HTML}")


if __name__ == "__main__":
    main()

 Rank           Company        Sector  Debt/EBITDA (x)  EBITDA/Interest (x)  EBITDA Margin (%)  FFO/Debt (%)  Revenue Growth (%)  Leverage Score  Coverage Score  Margin Score  FFO/Debt Score  Growth Score  Quant Weighted Score  Business Profile Score  Blended Score  Notching Adj.                           Notching Reason Final Rating      Rating Grade
    1   Delta Tech Corp Tech / Growth             0.61                24.75               30.0         135.0                15.0               1               1             2               1             2                  1.45                       3           1.92              0                                                     Aa  Investment Grade
    2 Alpha Industrials    Industrial             1.71                11.67               25.0          45.0                 7.7               2               2             3               2             3                  2.30                       3           2.51              0            